# Historical mineral prices: source coverage

Run all cells from your Semiconductors-DSS clone. The checklist reads the CMO workbook and USGS commodity CSV directly. It shows which minerals can currently support a monthly volatility target.

In [1]:
import csv
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path
from xml.etree import ElementTree as ET
from zipfile import ZipFile

## Inputs

Set REPO_ROOT if the notebook kernel runs outside your clone. Edit MINERALS to change the scope.

In [14]:
# Set to Path("/path/to/Semiconductors-DSS") if the clone is elsewhere.
REPO_ROOT = None
if REPO_ROOT is None:
    working_dir = Path.cwd().resolve()
    REPO_ROOT = next(
        (folder for folder in (working_dir, *working_dir.parents) if (folder / "Data_econ").is_dir()),
        None,
    )
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook inside your Semiconductors-DSS clone or set REPO_ROOT above.")

DATA_ECON = REPO_ROOT / "Data_econ"
WORLD_BANK_FILE = DATA_ECON / "CMO" / "CMO-Historical-Data-Monthly.xlsx"
USGS_FILE = DATA_ECON / "MCS2026_Commodities_Data.csv"


# Edit this list to change the minerals included in the three outputs.
MINERALS = ("Cobalt", "Gallium", "Germanium", "Lithium", "Silicon", "Tantalum", "Tin", "Tungsten")

MINERALS = ("Cobalt", "Gallium", "Germanium", "Lithium", "Silicon", "Tantalum", "Tin", "Tungsten")
NS = {"x": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
REL_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"

FileNotFoundError: Open the notebook inside your Semiconductors-DSS clone or set REPO_ROOT above.

In [3]:
def monthly_sheet(path):
    """Read the Monthly Prices worksheet using only the Python standard library."""
    with ZipFile(path) as book:
        workbook = ET.fromstring(book.read("xl/workbook.xml"))
        sheet = workbook.find(".//x:sheet[@name='Monthly Prices']", NS)
        if sheet is None:
            raise ValueError("World Bank workbook has no Monthly Prices worksheet")
        sheet_id = sheet.attrib[f"{{{REL_NS}}}id"]
        relationships = ET.fromstring(book.read("xl/_rels/workbook.xml.rels"))
        target = next(rel.attrib["Target"] for rel in relationships if rel.attrib["Id"] == sheet_id)
        sheet_path = target.lstrip("/") if target.startswith("/") else f"xl/{target}"
        data = ET.fromstring(book.read(sheet_path))
        strings = []
        if "xl/sharedStrings.xml" in book.namelist():
            shared = ET.fromstring(book.read("xl/sharedStrings.xml"))
            strings = ["".join(t.text or "" for t in item.findall(".//x:t", NS)) for item in shared]

    rows = {}
    for row in data.findall(".//x:row", NS):
        cells = {}
        for cell in row.findall("x:c", NS):
            column = cell.attrib["r"].rstrip("0123456789")
            value = cell.findtext("x:v", "", NS)
            if cell.get("t") == "s":
                value = strings[int(value)]
            elif cell.get("t") == "inlineStr":
                value = "".join(t.text or "" for t in cell.findall(".//x:t", NS))
            cells[column] = value.strip()
        rows[int(row.attrib["r"])] = cells
    return rows




## Eight-mineral coverage checklist

Months without a price are counted across the full CMO calendar. USGS counts are price records, so a mineral can have several records per year for different product forms or markets.

In [4]:
# Match the World Bank header to the project minerals and count numeric observations.
sheet = monthly_sheet(WORLD_BANK_FILE)
columns = {label: col for col, label in sheet[5].items() if label in MINERALS}
months = {}
for row in sheet.values():
    period = row.get("A", "")
    if re.fullmatch(r"[0-9]{4}M(0[1-9]|1[0-2])", period):
        if period in months:
            raise ValueError(f"Duplicate World Bank month: {period}")
        months[period] = row
if not months:
    raise ValueError("No dated observations in the World Bank worksheet")

def positive_number(raw_value):
    try:
        value = Decimal(raw_value.replace(",", "").strip())
    except InvalidOperation:
        return False
    return value.is_finite() and value > 0

annual = {mineral: [] for mineral in MINERALS}
with USGS_FILE.open(encoding="cp1252", newline="") as source:
    for row in csv.DictReader(source):
        mineral = row["MCS chapter"].title()
        if row["Statistics"] == "Price" and mineral in annual:
            annual[mineral].append(row)

coverage = []
for mineral in MINERALS:
    col = columns.get(mineral)
    observed = sorted(period for period, row in months.items()
                      if col and positive_number(row.get(col, "")))
    annual_rows = annual[mineral]
    numeric = [row for row in annual_rows if positive_number(row["Value"])]
    years = sorted({row["Year"] for row in numeric})
    series = {(row["Statistics_detail"], row["Unit"]) for row in annual_rows}
    coverage.append({
        "mineral": mineral,
        "cmo_unit": sheet[6].get(col, ""),
        "monthly_observations": len(observed),
        "first_month": observed[0].replace("M", "-") if observed else "",
        "last_month": observed[-1].replace("M", "-") if observed else "",
        "months_without_price": len(months) - len(observed),
        "usgs_first_year": years[0] if years else "",
        "usgs_last_year": years[-1] if years else "",
        "usgs_numeric_records": len(numeric),
        "usgs_missing_records": len(annual_rows) - len(numeric),
        "usgs_distinct_series": len(series),
        "coverage": "Monthly + annual" if observed and numeric else
                    "Monthly only" if observed else "Annual only" if numeric else "No prices",
    })


In [5]:
from IPython.display import HTML, display
from html import escape

headers = list(coverage[0])
header = "<tr>" + "".join(f"<th>{escape(key)}</th>" for key in headers) + "</tr>"
rows = [
    "<tr>" + "".join(f"<td>{escape(str(row[key]))}</td>" for key in headers) + "</tr>"
    for row in coverage
]
display(HTML("<table>" + header + "".join(rows) + "</table>"))

mineral,cmo_unit,monthly_observations,first_month,last_month,months_without_price,usgs_first_year,usgs_last_year,usgs_numeric_records,usgs_missing_records,usgs_distinct_series,coverage
Cobalt,,0,,,802,2021,2025,10,0,2,Annual only
Gallium,,0,,,802,2021,2025,5,0,1,Annual only
Germanium,,0,,,802,2021,2025,10,0,2,Annual only
Lithium,,0,,,802,2021,2025,5,0,1,Annual only
Silicon,,0,,,802,2021,2025,11,4,3,Annual only
Tantalum,,0,,,802,2021,2025,5,0,1,Annual only
Tin,($/mt),802,1960-01,2026-10,0,2021,2025,10,0,2,Monthly + annual
Tungsten,,0,,,802,2021,2025,5,0,1,Annual only


## Inspect an annual series

Compare definitions and units before choosing one series for a mineral. Annual values stay at annual frequency.

In [6]:
# Change this name to inspect the annual price definitions for another mineral.
MINERAL_TO_INSPECT = "Germanium"
for definition, unit in sorted({(r["Statistics_detail"], r["Unit"])
                                for r in annual[MINERAL_TO_INSPECT]}):
    print(f"{definition} [{unit}]")

Price, annual average, dollars per kilogram: Germanium dioxide [dollars per kilogram]
Price, annual average, dollars per kilogram: Germanium metal [dollars per kilogram]
